# 117 — Human-in-the-loop y aprobaciones

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**HITL:** ciertos pasos requieren decisión humana EN tiempo de ejecución (distinto de
on-the-loop: supervisión a posteriori). El `ask` de la matriz de permisos (116) se
materializa con el patrón **interrupt/resume**: el runtime suspende el bucle en un
punto consistente, persiste un checkpoint (115) con la acción propuesta y su
justificación, espera el veredicto (minutos u horas — el proceso puede morir y
renacer) y reanuda: **aprobar** (ejecutar), **editar** (ejecutar la versión
corregida) o **rechazar** (el motivo entra al contexto como observación y el agente
replantea).

### 📋 Contrato de la solicitud y calibración

El aprobador debe ver: (1) la acción EXACTA con argumentos literales, (2) el porqué
(objetivo + observaciones), (3) el impacto (clase de efecto, alcance, dry-run cuando
exista), (4) las alternativas si se rechaza. Si no puede decidir con eso, el defecto
es de la solicitud.

Criterio económico: `ask` se justifica cuando costo_error × prob_error > costo de
atención. Antídotos a la fatiga: aprobar por lotes/planes, umbrales cuantitativos,
y medir la tasa de rechazo (100 % de aprobaciones durante un mes = punto mal
calibrado). El laboratorio `workflow` ejecuta el esqueleto: `waiting_approval` es el
interrupt; `approved: true`, el veredicto; `completed` solo existe después.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Ejercicios

**Ejercicio 1 — El interrupt en la máquina de estados.** Ejecuta
`run_lab("workflow", seed=117)` y localiza en `events` el interrupt y el resume.
¿Qué transición sería IMPOSIBLE si `approved` fuera `false`, y qué estado nuevo
añadirías para representar el rechazo con motivo?

**Ejercicio 2 — Redacta la solicitud.** Un agente propone
`delete_branch(repo="app", branch="release-2024")` tras observar que no recibe commits
hace 18 meses. Redacta la solicitud de aprobación completa con las cuatro partes del
contrato (acción exacta, porqué, impacto, alternativas). Incluye qué mostraría el
dry-run.

**Ejercicio 3 — Coloca los puntos de aprobación.** Para un agente que gestiona el
correo de soporte con tools `read_inbox`, `draft_reply`, `send_reply`,
`add_to_blocklist`, `refund ≤ 30 €`, `refund > 30 €`: asigna allow/ask/deny,
justifica con el criterio económico y señala qué harías para evitar la fatiga si
`send_reply` genera 80 asks diarios.

**Ejercicio 4 — Simula interrupt/resume.** Implementa un mini-flujo: una función
`ejecutar_plan(pasos, aprobador)` donde cada paso marcado `"ask"` suspende, consulta al
aprobador (función que devuelve aprobar/editar/rechazar) y continúa según el veredicto.
Demuestra los tres veredictos con un plan de 3 pasos y muestra el log de auditoría.

In [ ]:
# TODO: ejecuta run_lab("workflow", seed=117)
# TODO: comprueba que el resultado incluya las claves 'kind' y 'evidence'
result = None


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 1: interrupt y resume en los eventos
result = run_lab("workflow", seed=117)
events = result["result"]["events"]
print(events)
interrupt = None   # ¿qué transición ENTRA al estado de espera?
resume = None      # ¿qué transición SALE de él?
# ¿qué estado añadirías para el rechazo con motivo?


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 4: interrupt/resume simulado
def ejecutar_plan(pasos, aprobador):
    log = []
    for paso in pasos:
        # paso = {"accion": "...", "politica": "allow" | "ask"}
        # si "ask": veredicto = aprobador(paso) -> {"verdict": ..., "motivo"/"edit": ...}
        # registra todo en log
        pass
    return log


## Reflexión

1. En el laboratorio la aprobación es síncrona e instantánea. ¿Qué dos piezas de
   ingeniería (clases 115 y 116) se vuelven imprescindibles cuando el veredicto tarda
   horas, y por qué?
2. ¿Por qué el veredicto "editar" evita un falso dilema, y qué debe registrar el log
   para que esa edición sea auditable?
3. Tu punto de aprobación lleva 3 meses con 100 % de aprobaciones. Da las dos lecturas
   posibles de esa métrica y qué acción tomarías en cada caso.